## 1、SummarizationMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization

API：https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware

作用：对历史消息列表进行摘要&总结，达到上下文压缩的效果。

原理：在达到触发条件时，调用大模型对历史消息进行摘要，将摘要的结果作为HumanMessage，放到消息列表最开始的位置。

### 1.1 参数说明

- model：用于摘要的模型，可以是模型名称，也可以是模型对象，如果传递的是模型名称，底层会调用init_chat_model初始化模型。
- trigger：摘要触发条件。是一个列表，每个元素对应一个条件，当任一条件满足时，触发摘要。
    - tokens：token的数量，历史token的累积数量达到该值时触发摘要。
    - messages：历史消息数量，历史消息条数达到该值时触发摘要。
    - fraction：上下文长度比例，历史token的累积数量达到模型的max_input_tokens * fraction时触发摘要。

    如果条件包含fraction，要求模型的profile包含max_input_tokens，Deepseek模型的profile为空，此时需要手动添加该配置项。Deepseek-V3.2的上下文长度为128K。

- keep： 摘要时保留的历史消息，支持三种条件，但是和trigger不同，keep同一时间只能接收一种条件。
    - tokens：摘要时保留的token数量。
    - messages：摘要时保留的历史消息条数。
    - fraction：摘要时保留max_input_tokens * fraction个token。

- token_counter：统计token数量的函数。默认使用LangChain提供的count_tokens_approximately，一般不用更改。对于纯文本消息，该函数的大致思路是先统计消息的字符数，也就是 len(字符串) ，然后再除以每
个token大致的字符数，转换为粗略的token数，再加一些额外开销。作为估算的token数。
- summary_prompt：摘要时的自定义提示词。该提示词需要包含{messages}占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。
- trim_token_to_summarize：摘要时历史消息的最大token数。如果历史消息token数大于该值，则会被裁减。默认为4000。如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。

### 1.2 举例1

测试trigger、keep参数

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # profile=custom_profile,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

agent = create_agent(
    model="deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2)  # 保留最新两条消息
        )
    ]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({
    "messages": messages
})

#print(response)
for msg in response['messages']:
    msg.pretty_print()

### 1.3 举例2

测试summary_prompt参数

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

custom_profile = {
    "max_input_tokens": 128_000
}

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # profile=custom_profile,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

agent = create_agent(
    model="deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2),  # 保留最新两条消息
            summary_prompt="对话历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

response = agent.invoke({
    "messages": messages
})

#print(response)
for msg in response['messages']:
    msg.pretty_print()

## 2、HumanInTheLoopMiddleware中间件

https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop

API：https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HumanInTheLoopMiddleware

HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在 工具调用前 中断Agent运行，等待用户对工具调用请求决策。可选的决策有： approve（同意执行） 、 edit（编辑调用配置后执行） 、 reject（拒绝执行） 。

### 2.1 参数说明

- interrupt_on：工具名和中断策略的映射。策略可以是True、False或InterruptOnConfig对象，精细控制决策选项。比如：
```
interrupt_on={
    "get_weather": True,
    "read_email_tool": False,
    "send_email_tool": {
        "allowed_decisions": ["approve", "reject"],
    },
}
```
True表示所有决策(approve, edit, reject) 都可以选择，相当于
```
"get_weather": {
    "allowed_decisions": ["approve", "edit", "reject"],
}
```
False表示不中断，即无需审批即可执行。

InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
① allowed_decisions 精细控制中断后允许的决策。
② description ：特定工具的中断描述信息，优先级高于description_prefix，后 者会更改 所有 工具中断的描述。

- description_prefix：自定义中断描述，默认为"Tool execution requires approval"。

### 2.2 举例1:调用前中断

注意：本例需要从中断的位置让Agent继续运行，这就需要用到短期记忆，这里先使用即可。

创建Agent时通过checkpointer参数启用了短期记忆，在调用时通过传递相同的config加载记忆。

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from rich import print as rprint

from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断"
                }
            },
            description_prefix="中断了"
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气，"
            "查询今日新闻，"
            "查看ID为 'sk2131421' 的邮件内容，"
            "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'，"
            "同时做这四件事"
                     )
        ]
    },
    config=config,
)

rprint(response)

### 2.3 举例2:指明工具调用请求决策

In [ ]:
from langgraph.types import Command

weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "广州市", "is_forcast": True}
    }
}

new_decision = {
    "type": "approve"
}

send_email_decision = {
    "type": "approve"
}

decisions = {
    "decisions": []
}

interrupts = response.get("__interrupt__", [])
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(new_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config,
    )

    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()